# Parte 3 — Ecuaciones elípticas
## 3.8 Métodos de energía y principio de Dirichlet
### 3.8.01 Formulación variacional, coercividad, convexidad y Euler-Lagrange

Este notebook cubre el apartado **3.8** del temario oficial:

> Métodos de energía y el principio de Dirichlet.

## Relación con las notas manuscritas

Las notas disponibles terminan en la página 9 con función de Green y sus
estimaciones. No contienen una sección manuscrita sobre métodos variacionales.
Por tanto, esta unidad se marca íntegramente como

**Complemento para cerrar el temario oficial.**

Se conserva la convención ya utilizada para Poisson:

$$
-\Delta u=f.
$$

## Contenido

1. espacios $H^1(\Omega)$ y $H_0^1(\Omega)$ al nivel estrictamente necesario;
2. desigualdad de Poincaré;
3. formulación débil;
4. funcional de energía;
5. coercividad y convexidad estricta;
6. existencia por el método directo;
7. ecuación de Euler-Lagrange;
8. unicidad y estimación de energía;
9. principio de Dirichlet para datos no homogéneos;
10. descenso numérico de energía en coordenadas espectrales.

## Estado de las fuentes

Las capturas antes enlazadas de las notas, del temario y del Examen General 2026-1 no están incluidas en el repositorio. El desarrollo que sigue es autocontenido. Para cotejar el enunciado oficial debe consultarse el PDF original fuera de este repositorio; no se sustituye aquí por una imagen inventada.


# Simulaciones y visualizaciones

Las celdas se ejecutan directamente. No existe una bandera `VIDEO=True`.

Se incluyen:

1. descenso de energía en una base seno;
2. animación de la aproximación a la solución exacta;
3. evolución del funcional y del error;
4. paisaje estrictamente convexo sobre una dirección de prueba;
5. barrido numérico de la desigualdad de Poincaré.

CuPy/CUDA se usa automáticamente cuando está disponible.

In [ ]:
from __future__ import annotations

import math
import shutil
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter, PillowWriter
from IPython.display import Video, Image, display

BASE = Path("..")
FIG_DIR = BASE / "figuras"
ANIM_DIR = BASE / "animaciones"
FIG_DIR.mkdir(parents=True, exist_ok=True)
ANIM_DIR.mkdir(parents=True, exist_ok=True)

GPU_AVAILABLE = False

try:
    import cupy as cp

    if cp.cuda.runtime.getDeviceCount() > 0:
        xp = cp
        GPU_AVAILABLE = True
        print("Backend numérico: CuPy/CUDA")
    else:
        raise RuntimeError("No se encontró un dispositivo CUDA.")
except Exception as exc:
    xp = np
    print("Backend numérico: NumPy/CPU")
    print("CuPy no disponible:", type(exc).__name__)


def to_cpu(array):
    if GPU_AVAILABLE:
        return cp.asnumpy(array)
    return np.asarray(array)


def save_and_display_animation(
    animation,
    stem,
    fps=60,
    dpi=150,
    bitrate=10000,
):
    """Guarda y muestra automáticamente una animación."""
    if shutil.which("ffmpeg"):
        output = ANIM_DIR / f"{stem}.mp4"
        writer = FFMpegWriter(
            fps=fps,
            bitrate=bitrate,
            metadata={"title": stem},
        )
        animation.save(output, writer=writer, dpi=dpi)
        display(Video(str(output), embed=True))
    else:
        output = ANIM_DIR / f"{stem}.gif"
        writer = PillowWriter(fps=min(fps, 35))
        animation.save(
            output,
            writer=writer,
            dpi=min(dpi, 110),
        )
        display(Image(filename=str(output)))

    print("Animación guardada en:", output.resolve())
    return output


def spectral_field(coefficients, sine_x, sine_y):
    """
    Reconstruye
        u(x,y) = sum_{m,n} c_{mn} sin(m pi x) sin(n pi y).
    """
    return sine_y.T @ coefficients.T @ sine_x


def spectral_energy(coefficients, forcing_coefficients, eigenvalues):
    """
    Energía en el cuadrado unitario.

    Como la norma L2 al cuadrado de cada producto seno es 1/4,
        J(c) = 1/8 sum lambda*c^2 - 1/4 sum f*c.
    """
    return (
        0.125 * xp.sum(eigenvalues * coefficients**2)
        - 0.25 * xp.sum(forcing_coefficients * coefficients)
    )

## Simulación 3.8.A — Descenso de energía espectral

En el cuadrado unitario consideramos

$$
-\Delta u=f,
\qquad
u=0
\quad\text{sobre }\partial(0,1)^2.
$$

La solución exacta se prescribe como

$$
u_*(x,y)
=
\sin(\pi x)\sin(\pi y)
+
0.30\sin(2\pi x)\sin(3\pi y)
-
0.20\sin(4\pi x)\sin(2\pi y).
$$

El dato $f$ se obtiene aplicando $-\Delta$.

En la base

$$
\sin(m\pi x)\sin(n\pi y),
$$

el funcional se vuelve una función cuadrática estrictamente convexa de los
coeficientes. Se aplica descenso de gradiente con paso estable.

In [ ]:
# ============================================================
# DESCENSO DE ENERGÍA EN COORDENADAS ESPECTRALES
# ============================================================

rng = np.random.default_rng(20260723)

M = 10
modes = xp.arange(1, M + 1, dtype=xp.float64)
m_grid, n_grid = xp.meshgrid(modes, modes, indexing="ij")

eigenvalues = (
    math.pi**2
    * (m_grid**2 + n_grid**2)
)

exact_coefficients = xp.zeros((M, M), dtype=xp.float64)
exact_coefficients[0, 0] = 1.0
exact_coefficients[1, 2] = 0.30
exact_coefficients[3, 1] = -0.20

forcing_coefficients = (
    eigenvalues * exact_coefficients
)

initial_cpu = 0.18 * rng.normal(size=(M, M))
initial_coefficients = xp.asarray(initial_cpu)

coefficients = initial_coefficients.copy()

lambda_max = float(to_cpu(xp.max(eigenvalues)))
step_size = 1.75 / lambda_max
n_steps = 1100

frame_steps = np.unique(
    np.concatenate(
        [
            np.array([0]),
            np.round(
                np.geomspace(1, n_steps, 190)
            ).astype(int),
        ]
    )
)

frame_step_set = set(int(value) for value in frame_steps)

coefficient_frames = []
energies = []
coefficient_errors = []
recorded_steps = []

for step in range(n_steps + 1):
    if step in frame_step_set:
        coefficient_frames.append(
            to_cpu(coefficients).astype(np.float32)
        )
        energies.append(
            float(
                to_cpu(
                    spectral_energy(
                        coefficients,
                        forcing_coefficients,
                        eigenvalues,
                    )
                )
            )
        )
        coefficient_errors.append(
            float(
                to_cpu(
                    xp.sqrt(
                        xp.sum(
                            (
                                coefficients
                                - exact_coefficients
                            ) ** 2
                        )
                    )
                )
            )
        )
        recorded_steps.append(step)

    gradient = (
        eigenvalues * coefficients
        - forcing_coefficients
    )
    coefficients = coefficients - step_size * gradient

coefficient_frames = np.asarray(coefficient_frames)
energies = np.asarray(energies)
coefficient_errors = np.asarray(coefficient_errors)
recorded_steps = np.asarray(recorded_steps)

exact_energy = float(
    to_cpu(
        spectral_energy(
            exact_coefficients,
            forcing_coefficients,
            eigenvalues,
        )
    )
)

print("Paso de descenso:", step_size)
print("Energía exacta:", exact_energy)
print("Energía final:", energies[-1])
print("Error final de coeficientes:", coefficient_errors[-1])

In [ ]:
# Reconstrucción espacial de los fotogramas.

grid_n = 320 if GPU_AVAILABLE else 190
x = xp.linspace(0.0, 1.0, grid_n)
y = xp.linspace(0.0, 1.0, grid_n)

sine_x = xp.sin(
    math.pi
    * modes[:, None]
    * x[None, :]
)
sine_y = xp.sin(
    math.pi
    * modes[:, None]
    * y[None, :]
)

field_frames = []

for coefficient_frame in coefficient_frames:
    field = spectral_field(
        xp.asarray(coefficient_frame),
        sine_x,
        sine_y,
    )
    field_frames.append(
        to_cpu(field).astype(np.float32)
    )

field_frames = np.asarray(field_frames)

exact_field = to_cpu(
    spectral_field(
        exact_coefficients,
        sine_x,
        sine_y,
    )
)

vmax = float(
    max(
        np.max(np.abs(field_frames)),
        np.max(np.abs(exact_field)),
    )
)

fig, ax = plt.subplots(figsize=(8.5, 7.0))

image = ax.imshow(
    field_frames[0],
    origin="lower",
    extent=[0.0, 1.0, 0.0, 1.0],
    interpolation="bilinear",
    vmin=-vmax,
    vmax=vmax,
)
fig.colorbar(image, ax=ax, label=r"$u_k(x,y)$")

ax.set_xlabel(r"$x$")
ax.set_ylabel(r"$y$")
ax.set_title("Descenso del funcional de energía")

status = ax.text(
    0.02,
    0.97,
    "",
    transform=ax.transAxes,
    va="top",
    bbox={"boxstyle": "round", "alpha": 0.8},
)


def update_descent(frame):
    image.set_data(field_frames[frame])
    status.set_text(
        rf"$k={recorded_steps[frame]}$"
        + "\n"
        + rf"$J={energies[frame]:.6f}$"
        + "\n"
        + rf"$\|c_k-c_*\|_2={coefficient_errors[frame]:.3e}$"
    )
    return image, status


animation = FuncAnimation(
    fig,
    update_descent,
    frames=len(field_frames),
    interval=1000.0 / 60.0,
    blit=False,
)

fig.tight_layout()

save_and_display_animation(
    animation,
    "03.8.A_descenso_de_energia",
    fps=60,
    dpi=165,
    bitrate=14000,
)

plt.close(fig)

In [ ]:
# Curvas de energía y error.

fig, ax = plt.subplots(figsize=(9, 5.7))
ax.semilogy(
    recorded_steps,
    energies - exact_energy,
    label=r"$J(u_k)-J(u_*)$",
)
ax.semilogy(
    recorded_steps,
    coefficient_errors,
    label=r"$\|c_k-c_*\|_2$",
)
ax.set_xlabel("iteración")
ax.set_ylabel("magnitud")
ax.set_title("Convergencia del descenso de energía")
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()

path = FIG_DIR / "03.8.A_convergencia_energia.png"
fig.savefig(path, dpi=220)
plt.show()
plt.close(fig)

print(path.resolve())

## Simulación 3.8.B — Paisaje estrictamente convexo

Tomamos la dirección

$$
\varphi(x,y)
=
\sin(5\pi x)\sin(4\pi y)
$$

y evaluamos

$$
s\longmapsto J(u_*+s\varphi).
$$

La curva debe ser una parábola con mínimo único en $s=0$.

In [ ]:
direction = xp.zeros_like(exact_coefficients)
direction[4, 3] = 1.0

s_values = np.linspace(-1.2, 1.2, 500)
landscape = []

for scalar in s_values:
    trial = (
        exact_coefficients
        + scalar * direction
    )
    landscape.append(
        float(
            to_cpu(
                spectral_energy(
                    trial,
                    forcing_coefficients,
                    eigenvalues,
                )
            )
        )
    )

landscape = np.asarray(landscape)

fig, ax = plt.subplots(figsize=(9, 5.7))
ax.plot(
    s_values,
    landscape,
)
ax.axvline(0.0, linestyle="--")
ax.axhline(exact_energy, linestyle=":")
ax.set_xlabel(r"$s$")
ax.set_ylabel(r"$J(u_*+s\varphi)$")
ax.set_title("Convexidad estricta del funcional")
ax.grid(True, alpha=0.3)
fig.tight_layout()

path = FIG_DIR / "03.8.B_paisaje_convexo.png"
fig.savefig(path, dpi=220)
plt.show()
plt.close(fig)

print(
    "Mínimo numérico en s =",
    s_values[np.argmin(landscape)],
)
print(path.resolve())

## Simulación 3.8.C — Desigualdad de Poincaré

Para

$$
v(x,y)
=
\sum_{m,n=1}^{M}
c_{mn}\sin(m\pi x)\sin(n\pi y),
$$

se tiene

$$
\|v\|_{L^2}^2
=
\frac14\sum_{m,n}c_{mn}^2,
$$

y

$$
\|\nabla v\|_{L^2}^2
=
\frac14\sum_{m,n}
\pi^2(m^2+n^2)c_{mn}^2.
$$

Por tanto,

$$
\frac{\|v\|_{L^2}^2}
{\|\nabla v\|_{L^2}^2}
\leq
\frac{1}{2\pi^2}.
$$

Se realiza un barrido aleatorio de coeficientes.

In [ ]:
# ============================================================
# BARRIDO NUMÉRICO DE POINCARÉ
# ============================================================

n_samples = 1_000_000 if GPU_AVAILABLE else 120_000
batch_size = 100_000 if GPU_AVAILABLE else 20_000

ratios_batches = []

remaining = n_samples

while remaining > 0:
    current = min(batch_size, remaining)

    random_coefficients = xp.random.standard_normal(
        (current, M, M)
    )

    l2_squared = 0.25 * xp.sum(
        random_coefficients**2,
        axis=(1, 2),
    )

    gradient_squared = 0.25 * xp.sum(
        eigenvalues[None, :, :]
        * random_coefficients**2,
        axis=(1, 2),
    )

    ratios_batches.append(
        to_cpu(
            l2_squared / gradient_squared
        )
    )

    remaining -= current

ratios = np.concatenate(ratios_batches)
poincare_bound = 1.0 / (2.0 * math.pi**2)

fig, ax = plt.subplots(figsize=(9, 5.7))
ax.hist(
    ratios,
    bins=90,
    density=True,
)
ax.axvline(
    poincare_bound,
    linestyle="--",
    label=r"$1/\lambda_1=1/(2\pi^2)$",
)
ax.set_xlabel(
    r"$\|v\|_{L^2}^2/\|\nabla v\|_{L^2}^2$"
)
ax.set_ylabel("densidad")
ax.set_title("Barrido numérico de la desigualdad de Poincaré")
ax.legend()
fig.tight_layout()

path = FIG_DIR / "03.8.C_poincare_barrido.png"
fig.savefig(path, dpi=220)
plt.show()
plt.close(fig)

print("Máximo observado:", float(np.max(ratios)))
print("Cota teórica:", poincare_bound)
print(path.resolve())

# 3.8.1 Espacios de energía

## **Complemento para cerrar el temario oficial**

Sea $\Omega\subset\mathbb R^n$ un dominio abierto y acotado.

## **Definición 3.8.1 (Espacio $H^1(\Omega)$).**

Se define

$$
H^1(\Omega)
=
\left\{
v\in L^2(\Omega):
\partial_i v\in L^2(\Omega)
\text{ en sentido débil},
\ i=1,\dots,n
\right\}.
$$

La norma es

$$
\|v\|_{H^1(\Omega)}^2
=
\|v\|_{L^2(\Omega)}^2
+
\|\nabla v\|_{L^2(\Omega)}^2.
$$

## **Definición 3.8.2 (Espacio $H_0^1(\Omega)$).**

Se define

$$
H_0^1(\Omega)
=
\overline{C_c^\infty(\Omega)}^{\,H^1}.
$$

Para dominios con frontera suficientemente regular, sus elementos son las
funciones de $H^1(\Omega)$ cuya traza sobre $\partial\Omega$ es cero.

### **Aclaración.**

No se desarrolla aquí la teoría general de trazas ni toda la teoría de Sobolev.
Sólo se usan:

1. que $H_0^1(\Omega)$ es un espacio de Hilbert;
2. la desigualdad de Poincaré;
3. la compacidad débil de sucesiones acotadas;
4. la semicontinuidad inferior de la norma.

## **Teorema 3.8.3 (Desigualdad de Poincaré).**

Sea $\Omega$ un dominio acotado. Existe una constante
$C_P=C_P(\Omega)>0$ tal que

$$
\|v\|_{L^2(\Omega)}
\leq
C_P\|\nabla v\|_{L^2(\Omega)}
$$

para todo $v\in H_0^1(\Omega)$.

En consecuencia,

$$
\|v\|_{H^1(\Omega)}
$$

y

$$
\|\nabla v\|_{L^2(\Omega)}
$$

son normas equivalentes en $H_0^1(\Omega)$.

### Comentario

La desigualdad falla en $H^1(\Omega)$ sin una normalización: las funciones
constantes tienen gradiente cero y norma $L^2$ positiva.

### Ejercicios — Sección 3.8.1

> **Ruta corta de estudio:** dos ejercicios representativos; este subtema se integra después en los problemas prioritarios.

1. Verifique Poincaré para una serie seno en $(0,\pi)$.

2. Explique por qué $H_0^1(\Omega)$ es el espacio natural para una condición
   homogénea de Dirichlet.


# 3.8.2 Formulación débil y funcional de energía

## **Definición 3.8.4 (Solución débil de Poisson-Dirichlet).**

Sea $f\in L^2(\Omega)$. Una función

$$
u\in H_0^1(\Omega)
$$

es una solución débil de

$$
\begin{cases}
-\Delta u=f,
& \text{en }\Omega,\\
u=0,
& \text{sobre }\partial\Omega
\end{cases}
$$

si

$$
\int_\Omega
\nabla u\cdot\nabla\varphi\,dx
=
\int_\Omega
f\varphi\,dx
$$

para toda

$$
\varphi\in H_0^1(\Omega).
$$

## **Definición 3.8.5 (Funcional de energía).**

Se define

$$
J_f(v)
=
\frac12
\int_\Omega|\nabla v|^2\,dx
-
\int_\Omega fv\,dx,
\qquad
v\in H_0^1(\Omega).
$$

El primer término es la energía de Dirichlet y el segundo representa el trabajo
de la fuente.

## **Proposición 3.8.6 (Coercividad).**

El funcional $J_f$ es coercivo en $H_0^1(\Omega)$:

$$
J_f(v)\to+\infty
$$

cuando

$$
\|v\|_{H_0^1}
=
\|\nabla v\|_{L^2}
\to\infty.
$$

Más precisamente,

$$
J_f(v)
\geq
\frac14\|\nabla v\|_{L^2}^2
-
C_P^2\|f\|_{L^2}^2.
$$

### Demostración

Por Cauchy-Schwarz y Poincaré,

$$
\left|
\int_\Omega fv\,dx
\right|
\leq
\|f\|_{L^2}\|v\|_{L^2}
\leq
C_P\|f\|_{L^2}
\|\nabla v\|_{L^2}.
$$

Usando la desigualdad de Young,

$$
ab
\leq
\frac14a^2+b^2,
$$

con

$$
a=\|\nabla v\|_{L^2},
\qquad
b=C_P\|f\|_{L^2},
$$

se obtiene

$$
J_f(v)
\geq
\frac14\|\nabla v\|_{L^2}^2
-
C_P^2\|f\|_{L^2}^2.
$$

$\square$

## **Proposición 3.8.7 (Convexidad estricta).**

Para $v,w\in H_0^1(\Omega)$ y $0<t<1$,

$$
J_f(tv+(1-t)w)
=
tJ_f(v)
+
(1-t)J_f(w)
-
\frac{t(1-t)}{2}
\|\nabla(v-w)\|_{L^2}^2.
$$

En particular, si $v\neq w$,

$$
J_f(tv+(1-t)w)
<
tJ_f(v)+(1-t)J_f(w).
$$

### Demostración

Se expande el cuadrado

$$
|t\nabla v+(1-t)\nabla w|^2
$$

y se usa la linealidad del término con $f$. Si
$\nabla(v-w)=0$, Poincaré implica $v=w$ en $H_0^1$.

$\square$

### Ejercicios — Sección 3.8.2

> **Ruta corta de estudio:** dos ejercicios representativos; este subtema se integra después en los problemas prioritarios.

1. Compruebe la identidad exacta de convexidad de la Proposición 3.8.7.

2. Muestre que el funcional deja de ser coercivo en $H^1(\Omega)$ si $f=0$ y no
   se fija ninguna condición de frontera.


# 3.8.3 Existencia por el método directo

## **Teorema 3.8.8 (Existencia de un minimizador).**

Sea $\Omega$ acotado y sea $f\in L^2(\Omega)$. Existe
$u\in H_0^1(\Omega)$ tal que

$$
J_f(u)
=
\inf_{v\in H_0^1(\Omega)}J_f(v).
$$

### Demostración añadida

Sea $\{v_k\}$ una sucesión minimizante:

$$
J_f(v_k)
\to
\inf_{H_0^1}J_f.
$$

La coercividad implica que $\{v_k\}$ está acotada en $H_0^1(\Omega)$. Como es
un espacio de Hilbert, existe una subsucesión y una función
$u\in H_0^1(\Omega)$ tales que

$$
v_k\rightharpoonup u
$$

débilmente en $H_0^1(\Omega)$.

La norma es semicontinua inferiormente:

$$
\|\nabla u\|_{L^2}^2
\leq
\liminf_{k\to\infty}
\|\nabla v_k\|_{L^2}^2.
$$

Además, el funcional lineal

$$
v\longmapsto\int_\Omega fv\,dx
$$

es continuo en $H_0^1$ por Poincaré, y por tanto es débilmente continuo. Así,

$$
J_f(u)
\leq
\liminf_{k\to\infty}J_f(v_k)
=
\inf_{H_0^1}J_f.
$$

Por tanto, $u$ es un minimizador.

$\square$

# 3.8.4 Ecuación de Euler-Lagrange

## **Teorema 3.8.9 (Minimizador si y sólo si solución débil).**

Sea $u\in H_0^1(\Omega)$. Son equivalentes:

1. $u$ minimiza $J_f$;
2. para toda $\varphi\in H_0^1(\Omega)$,

   $$
   \int_\Omega
   \nabla u\cdot\nabla\varphi\,dx
   =
   \int_\Omega
   f\varphi\,dx.
   $$

### Demostración

Supongamos que $u$ minimiza. Para
$\varphi\in H_0^1(\Omega)$ definimos

$$
F(t)
=
J_f(u+t\varphi).
$$

Como $F$ tiene un mínimo en $t=0$,

$$
F'(0)=0.
$$

Pero

$$
F'(0)
=
\int_\Omega
\nabla u\cdot\nabla\varphi\,dx
-
\int_\Omega
f\varphi\,dx.
$$

Esto prueba la ecuación débil.

Recíprocamente, supongamos que $u$ satisface la ecuación débil. Para
$v\in H_0^1(\Omega)$ escribimos

$$
v=u+w.
$$

Entonces

$$
\begin{aligned}
J_f(v)-J_f(u)
&=
\frac12\|\nabla w\|_{L^2}^2
+
\int_\Omega
\nabla u\cdot\nabla w\,dx
-
\int_\Omega fw\,dx\\
&=
\frac12\|\nabla w\|_{L^2}^2
\geq0.
\end{aligned}
$$

Por tanto, $u$ minimiza.

$\square$

## **Corolario 3.8.10 (Unicidad).**

El problema débil de Poisson-Dirichlet tiene una única solución.

### Primera demostración

El funcional es estrictamente convexo y, por tanto, tiene a lo más un
minimizador.

### Segunda demostración

Si $u_1$ y $u_2$ son soluciones, su diferencia $w$ satisface

$$
\int_\Omega
\nabla w\cdot\nabla\varphi\,dx
=
0
$$

para toda $\varphi\in H_0^1(\Omega)$. Tomando $\varphi=w$,

$$
\|\nabla w\|_{L^2}^2=0.
$$

Poincaré implica $w=0$.

## **Corolario 3.8.11 (Estimación de energía).**

La solución satisface

$$
\|\nabla u\|_{L^2}
\leq
C_P\|f\|_{L^2}.
$$

### Demostración

Tomando $\varphi=u$ en la formulación débil,

$$
\|\nabla u\|_{L^2}^2
=
\int_\Omega fu\,dx
\leq
C_P\|f\|_{L^2}\|\nabla u\|_{L^2}.
$$

$\square$

### Ejercicios — Secciones 3.8.3 y 3.8.4

> **Ruta corta de estudio:** dos ejercicios representativos; este subtema se integra después en los problemas prioritarios.

1. Explique dónde se usa la acotación de $\Omega$ en el método directo.

2. Complete todos los pasos de semicontinuidad inferior.


# 3.8.5 Principio de Dirichlet

## **Definición 3.8.12 (Clase afín con dato de frontera).**

Supongamos que el dato de frontera $g$ admite una extensión

$$
G\in H^1(\Omega).
$$

La clase admisible es

$$
\mathcal A_g
=
G+H_0^1(\Omega).
$$

Dos extensiones de $g$ producen la misma clase afín.

## **Teorema 3.8.13 (Principio de Dirichlet para funciones armónicas).**

Sea $u\in\mathcal A_g$. Son equivalentes:

1. $u$ es débilmente armónica:

   $$
   \int_\Omega
   \nabla u\cdot\nabla\varphi\,dx
   =
   0
   $$

   para toda $\varphi\in H_0^1(\Omega)$;

2. $u$ minimiza la energía de Dirichlet

   $$
   E(v)
   =
   \frac12
   \int_\Omega|\nabla v|^2\,dx
   $$

   sobre $\mathcal A_g$.

Además, para todo $v\in\mathcal A_g$,

$$
E(v)
=
E(u)
+
\frac12
\|\nabla(v-u)\|_{L^2}^2.
$$

### Demostración

Sea $w=v-u\in H_0^1(\Omega)$. Entonces

$$
\begin{aligned}
E(v)
&=
\frac12
\int_\Omega
|\nabla u+\nabla w|^2\,dx\\
&=
E(u)
+
\int_\Omega
\nabla u\cdot\nabla w\,dx
+
\frac12
\|\nabla w\|_{L^2}^2.
\end{aligned}
$$

La identidad débil anula el término cruzado. Esto prueba minimalidad y
unicidad. El recíproco se obtiene diferenciando
$E(u+t\varphi)$ en $t=0$.

$\square$

## **Teorema 3.8.14 (Principio variacional para Poisson con dato no homogéneo).**

En la clase $\mathcal A_g$, la solución débil de

$$
-\Delta u=f
$$

minimiza

$$
J_f(v)
=
\frac12
\int_\Omega|\nabla v|^2\,dx
-
\int_\Omega fv\,dx.
$$

Después de escribir

$$
v=G+w,
\qquad
w\in H_0^1(\Omega),
$$

el problema se reduce a minimizar un funcional coercivo y estrictamente convexo
sobre $H_0^1(\Omega)$.

### **Observación 3.8.15.**

El principio de Dirichlet no afirma que toda función continua con el dato de
frontera sea admisible. La clase natural es energética:

$$
\mathcal A_g\subset H^1(\Omega).
$$

La existencia clásica y la existencia variacional son problemas relacionados,
pero no idénticos. La regularidad elíptica permite pasar de una solución débil a
una solución clásica cuando $f$, $g$ y la frontera son suficientemente regulares.

### Ejercicios — Sección 3.8.5

> **Ruta corta de estudio:** dos ejercicios representativos; este subtema se integra después en los problemas prioritarios.

1. Demuestre que la clase $\mathcal A_g$ no depende de la extensión elegida.

2. Pruebe la identidad exacta de energía del Teorema 3.8.13.


# 3.8.6 Interpretación física y numérica

## **Proposición 3.8.16 (Descenso de energía).**

El flujo formal de gradiente de $J_f$ en la métrica $L^2$ es

$$
u_t
=
\Delta u+f,
$$

con condición homogénea de Dirichlet.

A lo largo de una solución suficientemente regular,

$$
\frac{d}{dt}J_f(u(t))
=
-
\int_\Omega|u_t|^2\,dx
\leq0.
$$

### Demostración formal

Como

$$
\frac{\delta J_f}{\delta u}
=
-\Delta u-f,
$$

se tiene

$$
u_t
=
-
\frac{\delta J_f}{\delta u}.
$$

Entonces

$$
\frac{d}{dt}J_f(u(t))
=
\left\langle
\frac{\delta J_f}{\delta u},
u_t
\right\rangle
=
-\|u_t\|_{L^2}^2.
$$

$\square$

### Interpretación

- La solución estacionaria del flujo satisface $-\Delta u=f$.
- La coercividad impide que una sucesión de energía acotada escape al infinito.
- La convexidad estricta impide mínimos múltiples.
- La ecuación de Euler-Lagrange identifica el mínimo con la EDP.
- El descenso espectral de la Simulación 3.8.A es una discretización de este
  principio.

### Ejercicios integradores — Métodos de energía

> **Ruta corta de estudio:** dos ejercicios representativos; este subtema se integra después en los problemas prioritarios.

1. Derive rigurosamente la identidad de disipación para una solución suave del
   flujo de gradiente.

2. Compare descenso explícito, descenso precondicionado y resolución directa en
   una base de funciones propias.


# Control de cobertura y estado del capítulo

## Contenido cubierto

- $H^1$ y $H_0^1$ al nivel necesario;
- desigualdad de Poincaré;
- formulación débil de Poisson;
- funcional de energía;
- coercividad;
- convexidad estricta;
- existencia por el método directo;
- ecuación de Euler-Lagrange;
- unicidad;
- estimación energética;
- principio de Dirichlet;
- dato no homogéneo mediante una clase afín;
- flujo de gradiente y descenso numérico GPU/CPU.

## Relación con las fuentes

La unidad completa un hueco explícito del temario oficial. Las notas manuscritas
disponibles no contienen este bloque, por lo que no se presenta ninguna parte
como transcripción literal.

## Pendiente inmediato

El siguiente notebook de la cola es

$$
\texttt{03.9.01\_Valores\_propios\_y\_armonicos\_esfericos.ipynb}.
$$

Se desarrollarán el problema espectral de Dirichlet, cociente de Rayleigh,
ortogonalidad, separación en coordenadas esféricas y armónicos esféricos.